# PWR17 2-D/3-D CSG Visualization Walkthrough

This notebook walks through the VERA-style 17x17 PWR assembly scaffold in `pwr17_assembly.py`. It shows the same product vocabulary used by the live pydetran CSG examples: fixed-grid slices, region-colored SVGs, sampled ray traces, and an output bundle.

The PWR17 example is currently a deterministic Python geometry scaffold for visualization and product-shape testing. It is not yet a native C++ CSG model and it does not use the new exact C++ tracker. The spheroid notebook is the live-backend counterpart.

Start Jupyter from a shell with the local Detran runtime on the path:

```bash
export DETRAN_ROOT=/home/robertsj/Research/detran
export PATH=$DETRAN_ROOT/install/bin:$PATH
export LD_LIBRARY_PATH=$DETRAN_ROOT/install/lib:$LD_LIBRARY_PATH
export PYTHONPATH=$DETRAN_ROOT/install/python:$PYTHONPATH
cd $DETRAN_ROOT/detran-examples/csg
jupyter notebook
```

The `LD_LIBRARY_PATH` setting needs to be present before the Python kernel starts.

In [ ]:
from pathlib import Path
import os
import sys

from IPython.display import SVG, Markdown, display

DETRAN_ROOT = Path(os.environ.get("DETRAN_ROOT", "/home/robertsj/Research/detran")).resolve()
EXAMPLE_DIR = DETRAN_ROOT / "detran-examples" / "csg"
INSTALL_PYTHON = DETRAN_ROOT / "install" / "python"

for path in (EXAMPLE_DIR, INSTALL_PYTHON):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import pwr17_assembly as pwr

OUTPUT_DIR = EXAMPLE_DIR / "notebook_outputs" / "pwr17"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Detran root: {DETRAN_ROOT}")
print(f"Notebook outputs: {OUTPUT_DIR}")

## Region Legend

The slice and ray SVGs carry `data-region-id` metadata and use the same colors throughout the walkthrough.

In [ ]:
def markdown_table(headers, rows):
    table = ["| " + " | ".join(headers) + " |"]
    table.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        table.append("| " + " | ".join(str(value) for value in row) + " |")
    return "\n".join(table)

legend_rows = []
for region_id, name in sorted(pwr.REGION_NAMES.items()):
    color = pwr.REGION_COLORS[region_id]
    swatch = f"<span style='display:inline-block;width:1.2em;height:1.2em;background:{color};border:1px solid #555'></span>"
    legend_rows.append([region_id, name, swatch, color])

display(Markdown(markdown_table(["id", "region", "swatch", "hex"], legend_rows)))

## Geometry Constants

The dimensions are VERA-style assembly constants in centimeters. These are centralized in `PWR17Spec`, which also defines the guide-tube and spacer-grid axial positions.

In [ ]:
spec = pwr.PWR17Spec()
display(Markdown(markdown_table(
    ["quantity", "value"],
    [
        ["pin pitch", f"{spec.pin_pitch:.4f} cm"],
        ["assembly pitch", f"{spec.assembly_pitch:.4f} cm"],
        ["total height", f"{spec.total_height:.4f} cm"],
        ["fuel stack", f"{spec.fuel_stack_bottom:.4f} to {spec.fuel_stack_top:.4f} cm"],
        ["fuel stack midplane", f"{spec.fuel_stack_midplane:.4f} cm"],
        ["guide tubes", len(pwr.GUIDE_TUBE_LOCATIONS)],
        ["instrument tube", pwr.INSTRUMENT_LOCATION],
    ],
)))

## Pin-Cell Exit Ramp

The single pin cell is the smallest visual unit: fuel, gap, clad, and moderator.

In [ ]:
pin_model = pwr.fuel_pin_2d_model()
pin_slice = pin_model.sample_slice(260, 260)

display(SVG(data=pwr.slice_to_svg(pin_slice, width=520, title="PWR17 fuel pin 2-D slice")))

In [ ]:
pin_rays = pin_model.trace_rays(angle_degrees=0.0, number_rays=1, sample_count=800)
display(Markdown(f"One centerline ray was collapsed into **{pin_rays.number_segments}** sampled region segments."))
display(SVG(data=pwr.ray_trace_to_svg(pin_rays, width=520, stroke_width=4.0, title="Fuel pin sampled centerline trace")))

## 2-D Assembly Slices

The 2-D assembly plane shows the full 17x17 pin lattice, guide-tube pattern, central instrument tube, and the effect of control insertion in guide tubes.

In [ ]:
for control_position in (0.0, 1.0):
    assembly_2d = pwr.assembly_2d_model(control_position=control_position)
    product = assembly_2d.sample_slice(220, 220)
    display(Markdown(f"### 2-D assembly, control_position={control_position:.1f}"))
    display(SVG(data=pwr.slice_to_svg(
        product,
        width=640,
        title=f"PWR17 assembly 2-D, control={control_position:.1f}",
    )))

## 3-D Assembly Axial Planes

The 3-D scaffold classifies the same x-y coordinates differently as `z` moves through nozzles, fuel, spacer-grid bands, and control poison.

In [ ]:
assembly_3d = pwr.assembly_3d_model(control_position=1.0, include_spacer_grids=True)
planes = [
    ("bottom nozzle", 0.5 * spec.bottom_nozzle_height),
    ("fuel stack midplane", spec.fuel_stack_midplane),
    ("spacer grid band", 75.2),
    ("top nozzle", spec.total_height - 0.5 * spec.top_nozzle_height),
]

for label, z in planes:
    product = assembly_3d.sample_slice(220, 220, z=z)
    display(Markdown(f"### {label}, z={z:.3f} cm"))
    display(SVG(data=pwr.slice_to_svg(
        product,
        width=640,
        title=f"PWR17 {label}, z={z:.3f} cm",
    )))

## Sampled Ray Traces

These ray traces are inspection products. They are sampled along straight lines and merged into region segments, which gives a track-like product for visualization and schema work.

In [ ]:
ray_rows = []
for label, z, angle in (
    ("midplane", spec.fuel_stack_midplane, 45.0),
    ("spacer grid", 75.2, 12.0),
):
    rays = assembly_3d.trace_rays(
        z=z,
        angle_degrees=angle,
        number_rays=35,
        sample_count=700,
    )
    ray_rows.append([label, f"{z:.3f}", angle, rays.number_rays, rays.number_segments])
    display(Markdown(f"### {label} rays, angle={angle:.1f} degrees"))
    display(SVG(data=pwr.ray_trace_to_svg(
        rays,
        width=640,
        stroke_width=1.6,
        title=f"PWR17 {label} sampled rays",
    )))

display(Markdown(markdown_table(
    ["plane", "z [cm]", "angle", "rays", "segments"],
    ray_rows,
)))

## Write a Compact SVG Bundle

This cell creates a ready-to-inspect output directory with the standard slice and ray products. It uses SVG slices to avoid optional Pillow and skips HDF5 unless you enable it later.

In [ ]:
bundle_dir = OUTPUT_DIR / "bundle"
written = pwr.write_example_outputs(
    bundle_dir,
    control_position=1.0,
    nx=180,
    ny=180,
    rays=True,
    number_rays=25,
    ray_samples=500,
    ray_angle=(0.0, 45.0),
    hdf5=False,
    slice_format="svg",
)

for path in written:
    print(path.relative_to(EXAMPLE_DIR))

## Optional HDF5 Product

If `h5py` is installed, this writes the same semantic products to a small HDF5 file for downstream readers.

In [ ]:
try:
    import h5py  # noqa: F401
except ImportError:
    print("h5py is not installed; skipping optional HDF5 output.")
else:
    hdf5_dir = OUTPUT_DIR / "hdf5"
    written = pwr.write_example_outputs(
        hdf5_dir,
        control_position=0.75,
        nx=80,
        ny=80,
        rays=True,
        number_rays=9,
        ray_samples=250,
        ray_angle=(45.0,),
        hdf5=True,
        slice_format="svg",
    )
    for path in written:
        print(path.relative_to(EXAMPLE_DIR))